# 数据预处理

In [ ]:
import json
import os

import xxhash
import tqdm
import logging
import networkx as nx
from tqdm import tqdm
import time
import datetime

## 变量定义

In [ ]:
VERBOSE = False # 是否打印详细信息
NO_ENCODE = False # 是否不编码（哈希映射）
STATS = False # 是否统计时间戳（基于最小时间的偏差）
JIFFIES = True # 是否提取时间戳
BIDIRECTION = False # 是否双向图

# 样本数量
NORMAL_NUMS = 125 # <=125
ATTACK_NUMS = 25 # <=25

# 结点类型
valid_node_type = ['file', 'process_memory', 'task', 'mmaped_file', 'path', 'socket', 'address', 'link']

# 文件路径
raw_data_dir = "../data/wget/raw/"
processed_data_dir = "../data/CICAPT_IIOT/processed/"
final_data_dir = "../data/CICAPT_IIOT/final/type/"
# final_data_dir = "../data/CICAPT_IIOT/final/operation/"

## Json数据解析

首先解析所有结点，生成node_map，然后根据node_map解析所有边，将溯源图以边的形式输出，并保存在文件中

处理后的边文件保存在processed_data_dir中，格式为：

`<源节点ID>\t<目标节点ID>\t<源类型>:<目标类型>:<边类型>:<边逻辑时间戳>`


#### 结点解析

In [ ]:
def parse_nodes(json_string, node_map):
    """解析CamFlow JSON字符串中的节点("activity"或"entity")。
    解析结果存入@node_map字典，该字典将CamFlow分配的节点UID映射到表示节点类型的哈希值(str类型)。"""
    json_object = None
    try:
        # use "ignore" if non-decodeable exists in the @json_string
        json_object = json.loads(json_string)
    except Exception as e:
        print("Exception ({}) occurred when parsing a node in JSON:".format(e))
        print(json_string)
        exit(1)
    if "activity" in json_object:
        activity = json_object["activity"]
        for uid in activity:
            if not uid in node_map:  # only parse unseen nodes
                if "prov:type" not in activity[uid]:
                    # a node must have a type.
                    # record this issue if logging is turned on
                    if VERBOSE:
                        logging.debug("skipping a problematic activity node with no 'prov:type': {}".format(uid))
                else:
                    node_map[uid] = activity[uid]["prov:type"]

    if "entity" in json_object:
        entity = json_object["entity"]
        for uid in entity:
            if not uid in node_map:
                if "prov:type" not in entity[uid]:
                    if VERBOSE:
                        logging.debug("skipping a problematic entity node with no 'prov:type': {}".format(uid))
                else:
                    node_map[uid] = entity[uid]["prov:type"]

#### 解析文件

In [ ]:
def parse_all_nodes(filename, node_map):
    """解析CamFlow数据中的所有节点。@filename是数据文件路径，@node_map存储节点到其哈希属性的映射"""
    description = '\x1b[6;30;42m[STATUS]\x1b[0m Parsing nodes in CamFlow data from {}'.format(filename)
    pb = tqdm(desc=description, mininterval=1.0, unit=" recs")
    with open(filename, 'r') as f:
        # each line in CamFlow data could contain multiple
        # provenance nodes, we call @parse_nodes routine.
        for line in f:
            pb.update()  # for progress tracking
            parse_nodes(line, node_map)
    f.close()
    pb.close()

#### 边解析

In [ ]:
def hashgen(l):
    """从列表中生成单个哈希值。@l是一个字符串列表，可以是节点/边的属性。
    此函数返回一个哈希后的整数值。"""
    hasher = xxhash.xxh64()
    for e in l:
        hasher.update(e)
    return hasher.intdigest()

# inputfile：CamFlow数据文件
# outputfile：输出log文件
def parse_all_edges(inputfile, outputfile, node_map, noencode):
    """从CamFlow数据文件@inputfile中解析所有边(包括时间戳)到@outputfile。在调用此函数前，应先调用parse_all_nodes
    来填充@node_map中所有CamFlow文件的节点。如果设置了@noencode，我们不会将CamFlow生成的原始UUID哈希为整数。
    此函数返回从CamFlow数据集中解析出的有效边的总数。

    输出的边列表每行格式如下(如果未设置-s参数):
        <源节点ID> \t <目标节点ID> \t <源类型>:<目标类型>:<边类型>:<边逻辑时间戳>
    如果设置了-s参数，每行格式如下:
        <源节点ID> \t <目标节点ID> \t <源类型>:<目标类型>:<边类型>:<边逻辑时间戳>:<时间戳统计信息>"""
    total_edges = 0
    smallest_timestamp = None
    # scan through the entire file to find the smallest timestamp from all the edges.
    # this step is only needed if we need to add some statistical information.
    if STATS:
        description = '\x1b[6;30;42m[STATUS]\x1b[0m Scanning edges in CamFlow data from {}'.format(inputfile)
        pb = tqdm(desc=description, mininterval=1.0, unit=" recs")
        with open(inputfile, 'r') as f:
            for line in f:
                pb.update()
                json_object = json.loads(line)

                if "used" in json_object:
                    used = json_object["used"]
                    for uid in used:
                        if "prov:type" not in used[uid]:
                            continue
                        if "cf:date" not in used[uid]:
                            continue
                        if "prov:entity" not in used[uid]:
                            continue
                        if "prov:activity" not in used[uid]:
                            continue
                        srcUUID = used[uid]["prov:entity"]
                        dstUUID = used[uid]["prov:activity"]
                        if srcUUID not in node_map:
                            continue
                        if dstUUID not in node_map:
                            continue
                        timestamp_str = used[uid]["cf:date"]
                        ts = time.mktime(datetime.datetime.strptime(timestamp_str, "%Y:%m:%dT%H:%M:%S").timetuple())
                        if smallest_timestamp == None or ts < smallest_timestamp:
                            smallest_timestamp = ts

                if "wasGeneratedBy" in json_object:
                    wasGeneratedBy = json_object["wasGeneratedBy"]
                    for uid in wasGeneratedBy:
                        if "prov:type" not in wasGeneratedBy[uid]:
                            continue
                        if "cf:date" not in wasGeneratedBy[uid]:
                            continue
                        if "prov:entity" not in wasGeneratedBy[uid]:
                            continue
                        if "prov:activity" not in wasGeneratedBy[uid]:
                            continue
                        srcUUID = wasGeneratedBy[uid]["prov:activity"]
                        dstUUID = wasGeneratedBy[uid]["prov:entity"]
                        if srcUUID not in node_map:
                            continue
                        if dstUUID not in node_map:
                            continue
                        timestamp_str = wasGeneratedBy[uid]["cf:date"]
                        ts = time.mktime(datetime.datetime.strptime(timestamp_str, "%Y:%m:%dT%H:%M:%S").timetuple())
                        if smallest_timestamp == None or ts < smallest_timestamp:
                            smallest_timestamp = ts

                if "wasInformedBy" in json_object:
                    wasInformedBy = json_object["wasInformedBy"]
                    for uid in wasInformedBy:
                        if "prov:type" not in wasInformedBy[uid]:
                            continue
                        if "cf:date" not in wasInformedBy[uid]:
                            continue
                        if "prov:informant" not in wasInformedBy[uid]:
                            continue
                        if "prov:informed" not in wasInformedBy[uid]:
                            continue
                        srcUUID = wasInformedBy[uid]["prov:informant"]
                        dstUUID = wasInformedBy[uid]["prov:informed"]
                        if srcUUID not in node_map:
                            continue
                        if dstUUID not in node_map:
                            continue
                        timestamp_str = wasInformedBy[uid]["cf:date"]
                        ts = time.mktime(datetime.datetime.strptime(timestamp_str, "%Y:%m:%dT%H:%M:%S").timetuple())
                        if smallest_timestamp == None or ts < smallest_timestamp:
                            smallest_timestamp = ts

                if "wasDerivedFrom" in json_object:
                    wasDerivedFrom = json_object["wasDerivedFrom"]
                    for uid in wasDerivedFrom:
                        if "prov:type" not in wasDerivedFrom[uid]:
                            continue
                        if "cf:date" not in wasDerivedFrom[uid]:
                            continue
                        if "prov:usedEntity" not in wasDerivedFrom[uid]:
                            continue
                        if "prov:generatedEntity" not in wasDerivedFrom[uid]:
                            continue
                        srcUUID = wasDerivedFrom[uid]["prov:usedEntity"]
                        dstUUID = wasDerivedFrom[uid]["prov:generatedEntity"]
                        if srcUUID not in node_map:
                            continue
                        if dstUUID not in node_map:
                            continue
                        timestamp_str = wasDerivedFrom[uid]["cf:date"]
                        ts = time.mktime(datetime.datetime.strptime(timestamp_str, "%Y:%m:%dT%H:%M:%S").timetuple())
                        if smallest_timestamp == None or ts < smallest_timestamp:
                            smallest_timestamp = ts

                if "wasAssociatedWith" in json_object:
                    wasAssociatedWith = json_object["wasAssociatedWith"]
                    for uid in wasAssociatedWith:
                        if "prov:type" not in wasAssociatedWith[uid]:
                            continue
                        if "cf:date" not in wasAssociatedWith[uid]:
                            continue
                        if "prov:agent" not in wasAssociatedWith[uid]:
                            continue
                        if "prov:activity" not in wasAssociatedWith[uid]:
                            continue
                        srcUUID = wasAssociatedWith[uid]["prov:agent"]
                        dstUUID = wasAssociatedWith[uid]["prov:activity"]
                        if srcUUID not in node_map:
                            continue
                        if dstUUID not in node_map:
                            continue
                        timestamp_str = wasAssociatedWith[uid]["cf:date"]
                        ts = time.mktime(datetime.datetime.strptime(timestamp_str, "%Y:%m:%dT%H:%M:%S").timetuple())
                        if smallest_timestamp == None or ts < smallest_timestamp:
                            smallest_timestamp = ts
        f.close()
        pb.close()

    # we will go through the CamFlow data (again) and output edgelist to a file
    output = open(outputfile, "w+")
    description = '\x1b[6;30;42m[STATUS]\x1b[0m Parsing edges in CamFlow data from {}'.format(inputfile)
    pb = tqdm(desc=description, mininterval=1.0, unit=" recs")
    with open(inputfile, 'r') as f:
        for line in f:
            pb.update()
            json_object = json.loads(line)

            if "used" in json_object:
                used = json_object["used"]
                for uid in used:
                    if "prov:type" not in used[uid]:
                        # an edge must have a type; if not,
                        # we will have to skip the edge. Log
                        # this issue if verbose is set.
                        if VERBOSE:
                            logging.debug("edge (used) record without type: {}".format(uid))
                        continue
                    else:
                        edgetype = "used"
                    # cf:id is used as logical timestamp to order edges
                    if "cf:id" not in used[uid]:
                        # an edge must have a logical timestamp;
                        # if not we will have to skip the edge.
                        # Log this issue if verbose is set.
                        if VERBOSE:
                            logging.debug("edge (used) record without logical timestamp: {}".format(uid))
                        continue
                    else:
                        timestamp = used[uid]["cf:id"]
                    if "prov:entity" not in used[uid]:
                        # an edge's source node must exist;
                        # if not, we will have to skip the
                        # edge. Log this issue if verbose is set.
                        if VERBOSE:
                            logging.debug(
                                "edge (used/{}) record without source UUID: {}".format(used[uid]["prov:type"], uid))
                        continue
                    if "prov:activity" not in used[uid]:
                        # an edge's destination node must exist;
                        # if not, we will have to skip the edge.
                        # Log this issue if verbose is set.
                        if VERBOSE:
                            logging.debug(
                                "edge (used/{}) record without destination UUID: {}".format(used[uid]["prov:type"],
                                                                                            uid))
                        continue
                    srcUUID = used[uid]["prov:entity"]
                    dstUUID = used[uid]["prov:activity"]
                    # both source and destination node must
                    # exist in @node_map; if not, we will
                    # have to skip the edge. Log this issue
                    # if verbose is set.
                    if srcUUID not in node_map:
                        if VERBOSE:
                            logging.debug(
                                "edge (used/{}) record with an unseen srcUUID: {}".format(used[uid]["prov:type"], uid))
                        continue
                    else:
                        srcVal = node_map[srcUUID]
                    if dstUUID not in node_map:
                        if VERBOSE:
                            logging.debug(
                                "edge (used/{}) record with an unseen dstUUID: {}".format(used[uid]["prov:type"], uid))
                        continue
                    else:
                        dstVal = node_map[dstUUID]
                    if "cf:date" not in used[uid]:
                        # an edge must have a timestamp; if
                        # not, we will have to skip the edge.
                        # Log this issue if verbose is set.
                        if VERBOSE:
                            logging.debug("edge (used) record without timestamp: {}".format(uid))
                        continue
                    else:
                        # we only record @adjusted_ts if we need
                        # to record stats of CamFlow dataset.
                        if STATS:
                            ts_str = used[uid]["cf:date"]
                            ts = time.mktime(datetime.datetime.strptime(ts_str, "%Y:%m:%dT%H:%M:%S").timetuple())
                            adjusted_ts = ts - smallest_timestamp
                    if "cf:jiffies" not in used[uid]:
                        # an edge must have a jiffies timestamp; if
                        # not, we will have to skip the edge.
                        # Log this issue if verbose is set.
                        if VERBOSE:
                            logging.debug("edge (used) record without jiffies: {}".format(uid))
                        continue
                    else:
                        # we only record @jiffies if
                        # the option is set
                        if JIFFIES:
                            jiffies = used[uid]["cf:jiffies"]
                    total_edges += 1
                    if noencode:
                        if STATS:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}:{}\n".format(srcUUID, dstUUID, srcVal, dstVal, edgetype, timestamp, adjusted_ts))
                        elif JIFFIES:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}:{}\n".format(srcUUID, dstUUID, srcVal, dstVal, edgetype, timestamp,
                                                                  jiffies))
                        else:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}\n".format(srcUUID, dstUUID, srcVal, dstVal, edgetype, timestamp))
                    else:
                        if STATS:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}:{}\n".format(hashgen([srcUUID]), hashgen([dstUUID]), srcVal, dstVal, edgetype, timestamp,
                                                                  adjusted_ts))
                        elif JIFFIES:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}:{}\n".format(hashgen([srcUUID]), hashgen([dstUUID]), srcVal, dstVal, edgetype, timestamp,
                                                                  jiffies))
                        else:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}\n".format(hashgen([srcUUID]), hashgen([dstUUID]), srcVal, dstVal, edgetype, timestamp))

            if "wasGeneratedBy" in json_object:
                wasGeneratedBy = json_object["wasGeneratedBy"]
                for uid in wasGeneratedBy:
                    if "prov:type" not in wasGeneratedBy[uid]:
                        if VERBOSE:
                            logging.debug("edge (wasGeneratedBy) record without type: {}".format(uid))
                        continue
                    else:
                        edgetype = "wasGeneratedBy"
                    if "cf:id" not in wasGeneratedBy[uid]:
                        if VERBOSE:
                            logging.debug("edge (wasGeneratedBy) record without logical timestamp: {}".format(uid))
                        continue
                    else:
                        timestamp = wasGeneratedBy[uid]["cf:id"]
                    if "prov:entity" not in wasGeneratedBy[uid]:
                        if VERBOSE:
                            logging.debug("edge (wasGeneratedBy/{}) record without source UUID: {}".format(
                                wasGeneratedBy[uid]["prov:type"], uid))
                        continue
                    if "prov:activity" not in wasGeneratedBy[uid]:
                        if VERBOSE:
                            logging.debug("edge (wasGeneratedBy/{}) record without destination UUID: {}".format(
                                wasGeneratedBy[uid]["prov:type"], uid))
                        continue
                    srcUUID = wasGeneratedBy[uid]["prov:activity"]
                    dstUUID = wasGeneratedBy[uid]["prov:entity"]
                    if srcUUID not in node_map:
                        if VERBOSE:
                            logging.debug("edge (wasGeneratedBy/{}) record with an unseen srcUUID: {}".format(
                                wasGeneratedBy[uid]["prov:type"], uid))
                        continue
                    else:
                        srcVal = node_map[srcUUID]
                    if dstUUID not in node_map:
                        if VERBOSE:
                            logging.debug("edge (wasGeneratedBy/{}) record with an unsen dstUUID: {}".format(
                                wasGeneratedBy[uid]["prov:type"], uid))
                        continue
                    else:
                        dstVal = node_map[dstUUID]
                    if "cf:date" not in wasGeneratedBy[uid]:
                        if VERBOSE:
                            logging.debug("edge (wasGeneratedBy) record without timestamp: {}".format(uid))
                        continue
                    else:
                        if STATS:
                            ts_str = wasGeneratedBy[uid]["cf:date"]
                            ts = time.mktime(datetime.datetime.strptime(ts_str, "%Y:%m:%dT%H:%M:%S").timetuple())
                            adjusted_ts = ts - smallest_timestamp
                    if "cf:jiffies" not in wasGeneratedBy[uid]:
                        if VERBOSE:
                            logging.debug("edge (wasGeneratedBy) record without jiffies: {}".format(uid))
                        continue
                    else:
                        if JIFFIES:
                            jiffies = wasGeneratedBy[uid]["cf:jiffies"]
                    total_edges += 1
                    if noencode:
                        if STATS:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}:{}\n".format(srcUUID, dstUUID, srcVal, dstVal, edgetype, timestamp,
                                                                  adjusted_ts))
                        elif JIFFIES:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}:{}\n".format(srcUUID, dstUUID, srcVal, dstVal, edgetype, timestamp,
                                                                  jiffies))
                        else:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}\n".format(srcUUID, dstUUID, srcVal, dstVal, edgetype, timestamp))
                    else:
                        if STATS:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}:{}\n".format(hashgen([srcUUID]), hashgen([dstUUID]), srcVal,
                                                                  dstVal, edgetype, timestamp,
                                                                  adjusted_ts))
                        elif JIFFIES:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}:{}\n".format(hashgen([srcUUID]), hashgen([dstUUID]), srcVal,
                                                                  dstVal, edgetype, timestamp,
                                                                  jiffies))
                        else:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}\n".format(hashgen([srcUUID]), hashgen([dstUUID]), srcVal, dstVal,
                                                               edgetype, timestamp))

            if "wasInformedBy" in json_object:
                wasInformedBy = json_object["wasInformedBy"]
                for uid in wasInformedBy:
                    if "prov:type" not in wasInformedBy[uid]:
                        if VERBOSE:
                            logging.debug("edge (wasInformedBy) record without type: {}".format(uid))
                        continue
                    else:
                        edgetype = "wasInformedBy"
                    if "cf:id" not in wasInformedBy[uid]:
                        if VERBOSE:
                            logging.debug("edge (wasInformedBy) record without logical timestamp: {}".format(uid))
                        continue
                    else:
                        timestamp = wasInformedBy[uid]["cf:id"]
                    if "prov:informant" not in wasInformedBy[uid]:
                        if VERBOSE:
                            logging.debug("edge (wasInformedBy/{}) record without source UUID: {}".format(
                                wasInformedBy[uid]["prov:type"], uid))
                        continue
                    if "prov:informed" not in wasInformedBy[uid]:
                        if VERBOSE:
                            logging.debug("edge (wasInformedBy/{}) record without destination UUID: {}".format(
                                wasInformedBy[uid]["prov:type"], uid))
                        continue
                    srcUUID = wasInformedBy[uid]["prov:informant"]
                    dstUUID = wasInformedBy[uid]["prov:informed"]
                    if srcUUID not in node_map:
                        if VERBOSE:
                            logging.debug("edge (wasInformedBy/{}) record with an unseen srcUUID: {}".format(
                                wasInformedBy[uid]["prov:type"], uid))
                        continue
                    else:
                        srcVal = node_map[srcUUID]
                    if dstUUID not in node_map:
                        if VERBOSE:
                            logging.debug("edge (wasInformedBy/{}) record with an unseen dstUUID: {}".format(
                                wasInformedBy[uid]["prov:type"], uid))
                        continue
                    else:
                        dstVal = node_map[dstUUID]
                    if "cf:date" not in wasInformedBy[uid]:
                        if VERBOSE:
                            logging.debug("edge (wasInformedBy) record without timestamp: {}".format(uid))
                        continue
                    else:
                        if STATS:
                            ts_str = wasInformedBy[uid]["cf:date"]
                            ts = time.mktime(datetime.datetime.strptime(ts_str, "%Y:%m:%dT%H:%M:%S").timetuple())
                            adjusted_ts = ts - smallest_timestamp
                    if "cf:jiffies" not in wasInformedBy[uid]:
                        if VERBOSE:
                            logging.debug("edge (wasInformedBy) record without jiffies: {}".format(uid))
                        continue
                    else:
                        if JIFFIES:
                            jiffies = wasInformedBy[uid]["cf:jiffies"]
                    total_edges += 1
                    if noencode:
                        if STATS:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}:{}\n".format(srcUUID, dstUUID, srcVal, dstVal, edgetype, timestamp,
                                                                  adjusted_ts))
                        elif JIFFIES:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}:{}\n".format(srcUUID, dstUUID, srcVal, dstVal, edgetype, timestamp,
                                                                  jiffies))
                        else:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}\n".format(srcUUID, dstUUID, srcVal, dstVal, edgetype, timestamp))
                    else:
                        if STATS:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}:{}\n".format(hashgen([srcUUID]), hashgen([dstUUID]), srcVal,
                                                                  dstVal, edgetype, timestamp,
                                                                  adjusted_ts))
                        elif JIFFIES:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}:{}\n".format(hashgen([srcUUID]), hashgen([dstUUID]), srcVal,
                                                                  dstVal, edgetype, timestamp,
                                                                  jiffies))
                        else:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}\n".format(hashgen([srcUUID]), hashgen([dstUUID]), srcVal, dstVal,
                                                               edgetype, timestamp))

            if "wasDerivedFrom" in json_object:
                wasDerivedFrom = json_object["wasDerivedFrom"]
                for uid in wasDerivedFrom:
                    if "prov:type" not in wasDerivedFrom[uid]:
                        if VERBOSE:
                            logging.debug("edge (wasDerivedFrom) record without type: {}".format(uid))
                        continue
                    else:
                        edgetype = "wasDerivedFrom"
                    if "cf:id" not in wasDerivedFrom[uid]:
                        if VERBOSE:
                            logging.debug("edge (wasDerivedFrom) record without logical timestamp: {}".format(uid))
                            continue
                    else:
                        timestamp = wasDerivedFrom[uid]["cf:id"]
                    if "prov:usedEntity" not in wasDerivedFrom[uid]:
                        if VERBOSE:
                            logging.debug("edge (wasDerivedFrom/{}) record without source UUID: {}".format(
                                wasDerivedFrom[uid]["prov:type"], uid))
                        continue
                    if "prov:generatedEntity" not in wasDerivedFrom[uid]:
                        if VERBOSE:
                            logging.debug("edge (wasDerivedFrom/{}) record without destination UUID: {}".format(
                                wasDerivedFrom[uid]["prov:type"], uid))
                        continue
                    srcUUID = wasDerivedFrom[uid]["prov:usedEntity"]
                    dstUUID = wasDerivedFrom[uid]["prov:generatedEntity"]
                    if srcUUID not in node_map:
                        if VERBOSE:
                            logging.debug("edge (wasDerivedFrom/{}) record with an unseen srcUUID: {}".format(
                                wasDerivedFrom[uid]["prov:type"], uid))
                        continue
                    else:
                        srcVal = node_map[srcUUID]
                    if dstUUID not in node_map:
                        if VERBOSE:
                            logging.debug("edge (wasDerivedFrom/{}) record with an unseen dstUUID: {}".format(
                                wasDerivedFrom[uid]["prov:type"], uid))
                        continue
                    else:
                        dstVal = node_map[dstUUID]
                    if "cf:date" not in wasDerivedFrom[uid]:
                        if VERBOSE:
                            logging.debug("edge (wasDerivedFrom) record without timestamp: {}".format(uid))
                        continue
                    else:
                        if STATS:
                            ts_str = wasDerivedFrom[uid]["cf:date"]
                            ts = time.mktime(datetime.datetime.strptime(ts_str, "%Y:%m:%dT%H:%M:%S").timetuple())
                            adjusted_ts = ts - smallest_timestamp
                    if "cf:jiffies" not in wasDerivedFrom[uid]:
                        if VERBOSE:
                            logging.debug("edge (wasDerivedFrom) record without jiffies: {}".format(uid))
                        continue
                    else:
                        if JIFFIES:
                            jiffies = wasDerivedFrom[uid]["cf:jiffies"]
                    total_edges += 1
                    if noencode:
                        if STATS:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}:{}\n".format(srcUUID, dstUUID, srcVal, dstVal, edgetype, timestamp,
                                                                  adjusted_ts))
                        elif JIFFIES:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}:{}\n".format(srcUUID, dstUUID, srcVal, dstVal, edgetype, timestamp,
                                                                  jiffies))
                        else:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}\n".format(srcUUID, dstUUID, srcVal, dstVal, edgetype, timestamp))
                    else:
                        if STATS:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}:{}\n".format(hashgen([srcUUID]), hashgen([dstUUID]), srcVal,
                                                                  dstVal, edgetype, timestamp,
                                                                  adjusted_ts))
                        elif JIFFIES:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}:{}\n".format(hashgen([srcUUID]), hashgen([dstUUID]), srcVal,
                                                                  dstVal, edgetype, timestamp,
                                                                  jiffies))
                        else:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}\n".format(hashgen([srcUUID]), hashgen([dstUUID]), srcVal, dstVal,
                                                               edgetype, timestamp))

            if "wasAssociatedWith" in json_object:
                wasAssociatedWith = json_object["wasAssociatedWith"]
                for uid in wasAssociatedWith:
                    if "prov:type" not in wasAssociatedWith[uid]:
                        if VERBOSE:
                            logging.debug("edge (wasAssociatedWith) record without type: {}".format(uid))
                        continue
                    else:
                        edgetype = "wasAssociatedWith"
                    if "cf:id" not in wasAssociatedWith[uid]:
                        if VERBOSE:
                            logging.debug("edge (wasAssociatedWith) record without logical timestamp: {}".format(uid))
                        continue
                    else:
                        timestamp = wasAssociatedWith[uid]["cf:id"]
                    if "prov:agent" not in wasAssociatedWith[uid]:
                        if VERBOSE:
                            logging.debug("edge (wasAssociatedWith/{}) record without source UUID: {}".format(
                                wasAssociatedWith[uid]["prov:type"], uid))
                        continue
                    if "prov:activity" not in wasAssociatedWith[uid]:
                        if VERBOSE:
                            logging.debug("edge (wasAssociatedWith/{}) record without destination UUID: {}".format(
                                wasAssociatedWith[uid]["prov:type"], uid))
                        continue
                    srcUUID = wasAssociatedWith[uid]["prov:agent"]
                    dstUUID = wasAssociatedWith[uid]["prov:activity"]
                    if srcUUID not in node_map:
                        if VERBOSE:
                            logging.debug("edge (wasAssociatedWith/{}) record with an unseen srcUUID: {}".format(
                                wasAssociatedWith[uid]["prov:type"], uid))
                        continue
                    else:
                        srcVal = node_map[srcUUID]
                    if dstUUID not in node_map:
                        if VERBOSE:
                            logging.debug("edge (wasAssociatedWith/{}) record with an unseen dstUUID: {}".format(
                                wasAssociatedWith[uid]["prov:type"], uid))
                        continue
                    else:
                        dstVal = node_map[dstUUID]
                    if "cf:date" not in wasAssociatedWith[uid]:
                        if VERBOSE:
                            logging.debug("edge (wasAssociatedWith) record without timestamp: {}".format(uid))
                        continue
                    else:
                        if STATS:
                            ts_str = wasAssociatedWith[uid]["cf:date"]
                            ts = time.mktime(datetime.datetime.strptime(ts_str, "%Y:%m:%dT%H:%M:%S").timetuple())
                            adjusted_ts = ts - smallest_timestamp
                    if "cf:jiffies" not in wasAssociatedWith[uid]:
                        if VERBOSE:
                            logging.debug("edge (wasAssociatedWith) record without jiffies: {}".format(uid))
                        continue
                    else:
                        if JIFFIES:
                            jiffies = wasAssociatedWith[uid]["cf:jiffies"]
                    total_edges += 1
                    if noencode:
                        if STATS:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}:{}\n".format(srcUUID, dstUUID, srcVal, dstVal, edgetype, timestamp,
                                                                  adjusted_ts))
                        elif JIFFIES:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}:{}\n".format(srcUUID, dstUUID, srcVal, dstVal, edgetype, timestamp,
                                                                  jiffies))
                        else:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}\n".format(srcUUID, dstUUID, srcVal, dstVal, edgetype, timestamp))
                    else:
                        if STATS:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}:{}\n".format(hashgen([srcUUID]), hashgen([dstUUID]), srcVal,
                                                                  dstVal, edgetype, timestamp,
                                                                  adjusted_ts))
                        elif JIFFIES:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}:{}\n".format(hashgen([srcUUID]), hashgen([dstUUID]), srcVal,
                                                                  dstVal, edgetype, timestamp,
                                                                  jiffies))
                        else:
                            output.write(
                                "{}\t{}\t{}:{}:{}:{}\n".format(hashgen([srcUUID]), hashgen([dstUUID]), srcVal, dstVal,
                                                               edgetype, timestamp))
    f.close()
    output.close()
    pb.close()
    return total_edges


#### 时间归一化

In [ ]:
def time_normalize(filename):
    # 1. 读取所有行，提取时间戳统计信息
    with open(filename, "r", encoding="utf-8") as f:
        lines = f.readlines()

    stats = []
    for line in lines:
        parts = line.strip().split("\t")
        fields = parts[2].split(":")
        stats.append(float(fields[-1]))  # 最后一个就是时间戳统计信息

    # 2. min-max归一化
    vmin, vmax = min(stats), max(stats)
    if vmax == vmin:
        norm_stats = [0.0] * len(stats)
    else:
        norm_stats = [(v - vmin) / (vmax - vmin) for v in stats]

    # 3. 重新写入文件，每行在末尾追加归一化时间
    with open(filename, "w", encoding="utf-8") as f:
        for line, norm in zip(lines, norm_stats):
            line = line.strip()
            new_line = f"{line}:{norm:.6f}\n"
            f.write(new_line)

## 构建networkx图

#### 构建图列表

从processed_data_dir文件中读取边数据，将源节点、目的结点、结点特征和边特征解析并用列表存储

In [ ]:
# threshold表示最多处理threshold条数据
def read_single_graph(file_name, threshold):
    graph = []
    edge_cnt = 0
    with open(file_name, 'r') as f:
        for line in f:
            try:
                edge = line.strip().split("\t")
                new_edge = [edge[0], edge[1]] # 源结点和目的结点id

                attributes = edge[2].strip().split(":") # 特征
                source_node_type = attributes[0] # 源节点特征（类型）
                destination_node_type = attributes[1] # 目的结点特征（类型）
                # edge_type = attributes[2] # 边特征（类型）
                edge_type = attributes[3] # 边特征（操作）
                edge_time = attributes[5] # 归一化的时间特征

                new_edge.append(source_node_type)
                new_edge.append(destination_node_type)
                new_edge.append(edge_type)
                new_edge.append(edge_time)

                graph.append(new_edge)
                edge_cnt += 1
            except:
                print("{}".format(line))
    f.close()
    graph.sort(key=lambda e: e[5])
    if len(graph) <= threshold:
        return graph
    else:
        return graph[:threshold]

#### 构建networkx图

利用图列表数据构建networkx图

In [ ]:
def process_graph(name, threshold):
    graph = read_single_graph(name, threshold)
    result_graph = nx.DiGraph()
    cnt = 0
    for num, edge in enumerate(graph):
        cnt += 1
        src, dst, src_type, dst_type, edge_type = edge[:5]
        edge_time = edge[-1]
        if src_type in valid_node_type and dst_type in valid_node_type:
            if not result_graph.has_node(src):
                result_graph.add_node(src, type=src_type)
            if not result_graph.has_node(dst):
                result_graph.add_node(dst, type=dst_type)
            if not result_graph.has_edge(src, dst):
                result_graph.add_edge(src, dst, type=edge_type, time=edge_time)
                if BIDIRECTION:
                    result_graph.add_edge(dst, src, type='reverse_{}'.format(edge_type))
    return cnt, result_graph

## 数据统计

#### 统计networkx图信息并存储为json文件

统计networkx图的边和结点的个数

读取所有结点和边的特征，分别存储为列表：
```python
type_data ={
    "node_type_list": node_type_list,
    "edge_type_list": edge_type_list,
    "node_type_dict": node_type_dict,
    "edge_type_dict": edge_type_dict,
}
```

将networkx图以node-link格式存储为json文件，文件目录为final_data_dir
> node-link 是 NetworkX 提供的一种通用图序列化格式（JSON 格式的一种），用来表示图的节点和边。
> 它的基本思想是：
> - nodes：用一个列表存所有节点，每个节点是一个字典，可以有各种属性。
> - links：用一个列表存所有边，每条边是一个字典，包含源节点、目标节点和属性。
> 
> 格式：
> ```json
> {
>     "nodes": [
>         {"id": 1, "type": "A"},
>         {"id": 2, "type": "B"},
>         ...
>     ],
>     "links": [
>         {"source": 1, "target": 2, "type": "AB"},
>         {"source": 2, "target": 3, "type": "BA"},
>         ...
>     ]
> }
> ```

In [ ]:
def format_graph(g, name, type_data):
    new_g = nx.DiGraph()
    node_map = {}
    node_cnt = 0
    for n in g.nodes:
        node_map[n] = node_cnt
        new_g.add_node(node_cnt, type=g.nodes[n]["type"])
        node_cnt += 1

    for e in g.edges:
        src_new = node_map[e[0]]
        dst_new = node_map[e[1]]
        edge_type = g.edges[e]["type"]
        # 读取 edge_time（之前在构图时写入的 'time'）
        edge_time = float(
            g.edges[e]["time"]
        )  # 如果没有则为 None；也可以用条件判断不写入
        # 写入到 new_g
        if edge_time is None:
            new_g.add_edge(src_new, dst_new, type=edge_type)
        else:
            new_g.add_edge(src_new, dst_new, type=edge_type, time=edge_time)

    for n in new_g.nodes:
        node_type = new_g.nodes[n]["type"]
        if not node_type in type_data["node_type_dict"]:
            type_data["node_type_list"].append(node_type)
            type_data["node_type_dict"][node_type] = 1
        else:
            type_data["node_type_dict"][node_type] += 1
    for e in new_g.edges:
        edge_type = new_g.edges[e]["type"]
        if not edge_type in type_data["edge_type_dict"]:
            type_data["edge_type_list"].append(edge_type)
            type_data["edge_type_dict"][edge_type] = 1
        else:
            type_data["edge_type_dict"][edge_type] += 1
    for n in new_g.nodes:
        new_g.nodes[n]["type"] = type_data["node_type_list"].index(
            new_g.nodes[n]["type"]
        )
    for e in new_g.edges:
        new_g.edges[e]["type"] = type_data["edge_type_list"].index(
            new_g.edges[e]["type"]
        )
    with open("{}.json".format(name), "w", encoding="utf-8") as f:
        json.dump(nx.node_link_data(new_g), f)

## 执行处理

生成processed_data和`networkx`的node-link格式json文件，统计networkx图的信息`type_data`

#### 执行前准备

In [ ]:
# 创建目录
if not os.path.exists(raw_data_dir):
    os.mkdir(raw_data_dir)
if not os.path.exists(processed_data_dir):
    os.mkdir(processed_data_dir)
if not os.path.exists(final_data_dir):
    os.mkdir(final_data_dir)

# 变量
cnt = 0
fname_list = []
sample_nums = NORMAL_NUMS + ATTACK_NUMS
attack_nums = ATTACK_NUMS

threshold = 10000000  # infinity

interaction_dict = []
graph_cnt = 0
result_graphs = []
node_type_list = []
edge_type_list = []
node_type_dict = {}
edge_type_dict = {}

#### 生成`processed`数据

In [ ]:
for i in range(sample_nums):
    if i < attack_nums:
        fname_list.append("wget-baseline-attack-" + str(i) + ".log")
    else:
        fname_list.append("wget-normal-" + str(i - attack_nums) + ".log")
        
for fname in fname_list:
    cnt += 1
    node_map = dict()
    parse_all_nodes(raw_data_dir + "/{}".format(fname), node_map)
    total_edges = parse_all_edges(
        raw_data_dir + "/{}".format(fname),
        processed_data_dir + "/{}.log".format(cnt),
        node_map,
        NO_ENCODE,
    )

#### 时间特征处理

In [ ]:
for i in tqdm(range(1, sample_nums + 1)):
    file_path = f"../data/wget/processed/{i}.log"
    time_normalize(file_path)

#### 生成`final`数据

In [ ]:
processed_data_dir = "../data/CICAPT_IIOT/processed/"
# final_data_dir = "../data/CICAPT_IIOT/final/type/"
final_data_dir = "../data/CICAPT_IIOT/final/operation/"

threshold = 10000000  # infinity

BIDIRECTION = False  # 是否双向图

valid_node_type = ["directory", "file", "link", "network socket", "unknown", "None"]

In [ ]:
if not os.path.exists(final_data_dir):
    os.mkdir(final_data_dir)

In [ ]:
result_graphs = []
node_type_list = []
edge_type_list = []
node_type_dict = {}
edge_type_dict = {}

type_data = {
    "node_type_list": node_type_list,
    "edge_type_list": edge_type_list,
    "node_type_dict": node_type_dict,
    "edge_type_dict": edge_type_dict,
}

line_cnt = 0
for i in tqdm(range(0, 64)):
    single_cnt, result_graph = process_graph(
        "{}{}.log".format(processed_data_dir, i + 1), threshold
    )
    format_graph(result_graph, "{}{}".format(final_data_dir, i), type_data)
    line_cnt += single_cnt

In [ ]:
print(line_cnt // 150)
print(len(node_type_list))
print(node_type_dict)
print(len(edge_type_list))
print(edge_type_dict)

# 构建DGL数据集

从node-link格式json文件读取networkx图，构建dgl数据集，并生成pkl文件

In [2]:
import pickle as pkl
import dgl
import networkx as nx
import json
from tqdm import tqdm
import os

## 变量定义

In [8]:
# networkx图（node-link json格式）文件的存储路径
# final_data_dir = "../data/CICAPT_IIOT/final/type"
final_data_dir = "../data/wget/final"

# 文件编号从0开始
# 0-24是attack数据
# 25-149是benign数据
attack_start_idx = 0
attack_end_idx = 9
benign_start_idx = 119
benign_end_idx = 138

# 用于训练还是测试
mode = 'test'

# pkl文件存储路径
benign_nums = benign_end_idx - benign_start_idx + 1
attack_nums = attack_end_idx - attack_start_idx + 1
networkx_graph_dir = f"../data/wget/networkx_graph/{mode}-{benign_nums}-{attack_nums}/"

# dgl数据集名称
pkl_file_name = "graphs"


## DGL数据对象

In [9]:
class WgetDataset(dgl.data.DGLDataset):
    def process(self):
        pass

    def __init__(self, name):
        super(WgetDataset, self).__init__(name=name)

        # 待处理的文件编号
        attack_indices = list(range(attack_start_idx, attack_end_idx + 1))
        benign_indices = list(range(benign_start_idx, benign_end_idx + 1))
        file_list = attack_indices + benign_indices

        self.graphs = []  # DGL图
        self.labels = []  # 标签

        print("Loading {} dataset...".format(name))
        for i in tqdm(file_list):
            idx = i
            # 用networkx图构建DGL图
            g = dgl.from_networkx(
                # 从json文件中读取networkx图
                nx.node_link_graph(
                    json.load(open("{}/{}.json".format(final_data_dir, str(idx))))
                ),
                node_attrs=["type"],
                edge_attrs=["type", "time"],
            )
            self.graphs.append(g)
            if idx in attack_indices:  # 恶意数据标为1
                self.labels.append(1)
            else:  # 良性数据标为0
                self.labels.append(0)

    # 返回元组，第一个元素是dgl图，第二个元素是标签
    def __getitem__(self, i):
        return self.graphs[i], self.labels[i]

    def __len__(self):
        return len(self.graphs)

## 构建DGL图并存为pkl文件

In [10]:
def create_pkl_data(name):
    if not os.path.exists(networkx_graph_dir):
        os.mkdir(networkx_graph_dir)

    file_path = os.path.join(networkx_graph_dir, f"{name}.pkl")

    if os.path.exists(file_path):
        print(f"{file_path} already exists.")
    else:
        print(f"Creating dataset and saving to {file_path} ...")
        raw_data = WgetDataset(name)
        with open(file_path, "wb") as f:
            pkl.dump(raw_data, f)

    return raw_data

## 执行处理

In [12]:
dataset = create_pkl_data(pkl_file_name)

Creating dataset and saving to ../data/wget/networkx_graph/test-20-10/graphs.pkl ...
Loading graphs dataset...


100%|██████████| 30/30 [00:37<00:00,  1.25s/it]


In [ ]:
dataset